In [2]:
from pybis import Openbis


def log_in(eln_url: str, eln_token: str):
    try:
        session = Openbis(eln_url, verify_certificates=False)
        session.set_token(eln_token)
    except ValueError:
        print("Session is no longer valid. Please check if the token is still valid.")
        session = None

    return session


openbis_url = "https://localhost.openbis.net/"
openbis_token = "$pat-admin-260305094034675xB175D8982B3F7ECC827B38FF09BC5F61"

# Connect to openBIS
session = log_in(openbis_url, openbis_token)

In [13]:
# session.new_property_type(
#     code="Test_PROP",
#     label="Test Property",
#     description="This is a test property",
#     dataType="VARCHAR",
#     multiValue=True,
# )
import sys

sys.path.append("/home/jovyan/apps/aiidalab-openbis/")
from src import utils
session.get_object_type(type = "ELECTRONICS").get_property_assignments().df.code.values

array(['NAME', 'DESCRIPTION', 'MAIN_CATEGORY', 'SUB_CATEGORY', 'MODEL',
       'SERIAL_NUMBER', 'EMPA_ID', 'OBJECT_STATUS', 'RECEIVE_DATE',
       'LOCATION', 'MANUFACTURER', 'COMMENTS'], dtype=object)

In [50]:
obj = session.get_object("20251210134330958-264")
obj_type = str(obj.type)
obj_type_props = session.get_object_type(type = obj_type).get_property_assignments().df.code.values
all_components = {}
for prop in obj_type_props:
    prop_type = session.get_property_type(prop).dataType
    if prop_type == "SAMPLE":
        prop = prop.lower()
        prop_objs = obj.props[prop]
        if prop_objs:
            if isinstance(prop_objs, list):
                for prop_obj_id in prop_objs:
                    prop_obj = session.get_object(prop_obj_id)
                    prop_obj_type = str(prop_obj.type)
                    if prop_obj_type not in all_components.keys():
                        all_components[prop_obj_type] = [prop_obj]
                    else:
                        all_components[prop_obj_type].append(prop_obj)
            else:
                prop_obj = session.get_object(prop_objs)
                prop_obj_type = str(prop_obj.type)
                if prop_obj_type not in all_components.keys():
                    all_components[prop_obj_type] = [prop_obj]
                else:
                    all_components[prop_obj_type].append(prop_obj)

print(all_components)


{'PERSON': [attribute            value
-------------------  -------------------------------------------------------
code                 PERS1845
permId               20251210135558901-2702
identifier           /LAB205_ADMINISTRATIVE/PEOPLE/PERS1845
type                 PERSON
project              /LAB205_ADMINISTRATIVE/PEOPLE
parents              []
children             []
components           []
space                LAB205_ADMINISTRATIVE
experiment           /LAB205_ADMINISTRATIVE/PEOPLE/LAB_205_PERSON_COLLECTION
tags                 []
registrator          admin
registrationDate     2025-12-10 14:55:59
modifier             admin
modificationDate     2025-12-10 14:56:00
container
frozen               False
frozenForComponents  False
frozenForChildren    False
frozenForParents     False
frozenForDataSets    False
metaData             {}
immutableData, attribute            value
-------------------  -------------------------------------------------------
code                 PERS1846
p

In [10]:
import utils
x = ['20251210135558901-2702', '20251210135600039-2704', '20251210134241407-179', '20251210134243839-181', '20251210134246021-183', '20251210134312678-225', '20251210134318290-241', '20251210134319148-243', '20251210134247105-185', '20251210134312864-226', '20251210134248117-187', '20251210134249176-189', '20251210134250569-191', '20251210134252356-193', '20251210134253582-195', '20251210134316359-233', '20251210134257124-199', '20251210134315001-230', '20251210134313904-228', '20260220123150813-3411', '20260220123156691-3413', '20260220123202590-3414', '20260220123143011-3410', '20260220123213667-3422', '20260220123208367-3417', '20260220121612784-3404', '20251210134300587-203', '20251210134258552-201', '20251210134301220-204', '20251210134302751-206', '20251210134304030-208', '20251210134307048-214', '20251210134221151-157', '20260220122253814-3405', '20260220122335132-3406', '20260220122419630-3407', '20260220122553276-3408', '20251210134238692-177', '20251210134309553-218', '20251210134235531-173', '20251210134311768-223', '20251210134236586-175', '20251210134310580-221', '20251210134233835-171', '20260220122647615-3409', '20251210135530968-2551', '20251210134333547-269']
for i in x:
    utils.get_openbis_objects(session, sample_ident = i)

In [12]:
import utils
new_action_object = utils.create_openbis_object(
    session,
    type="ENTRY",
    experiment="/DEFAULT/DEFAULT/DEFAULT_EXP_3",
)

utils.create_openbis_dataset(
    session,
    type = "OBSERVABLE",
    sample = new_action_object,
    files = ["/home/jovyan/apps/aiidalab-openbis/013a.cdxml"],
    props = {'name': 'Logs', 'description': 'Logs', 'components': ['20251210134241407-179', '20251210134319148-243', '20251210134243839-181']}
)

In [12]:
utils.get_openbis_datasets(session, type = "OBSERVABLE").df.permId.values

array(['20260227165257408-3444', '20260227165545580-3445'], dtype=object)

In [8]:
session.url

'https://localhost.openbis.net/'